In [44]:
from warnings import filterwarnings

import pandas as pd
import plotly.express as px
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
)
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    PolynomialFeatures, StandardScaler,
)

filterwarnings("ignore")

In [45]:
df = pd.read_csv('data/data.csv', index_col=0)

In [46]:
df.sample(5)

,Reason for absence,Month of absence,Day of the week,Seasons,Transportation expense,Distance from Residence to Work,Service time,Age,Work load Average/day,Hit target,Disciplinary failure,Education,Son,Social drinker,Social smoker,Pet,Weight,Height,Body mass index,Absenteeism time in hours
ID,,,,,,,,,,,,,,,,,,,,
11,19,9,4,1,289,36,13,33,294.217,81,0,1,2,1,0,1,90,172,30,24
18,1,7,2,1,330,16,4,28,275.312,98,0,2,0,0,0,0,84,182,25,8
34,28,8,3,1,118,10,10,37,249.797,93,0,1,0,0,0,0,83,172,28,4
34,23,9,3,4,118,10,10,37,241.476,92,0,1,0,0,0,0,83,172,28,2
28,25,2,5,2,225,26,9,28,264.249,97,0,1,1,0,0,2,69,169,24,3


In [47]:
df.isna().mean().round(2)
# No missing data

Reason for absence                 0.0
Month of absence                   0.0
Day of the week                    0.0
Seasons                            0.0
Transportation expense             0.0
Distance from Residence to Work    0.0
Service time                       0.0
Age                                0.0
Work load Average/day              0.0
Hit target                         0.0
Disciplinary failure               0.0
Education                          0.0
Son                                0.0
Social drinker                     0.0
Social smoker                      0.0
Pet                                0.0
Weight                             0.0
Height                             0.0
Body mass index                    0.0
Absenteeism time in hours          0.0
dtype: float64

In [48]:
fig = px.histogram(df, x='Absenteeism time in hours', title="Majority of workers are absent 5 hours or less")
fig.write_image('images/hours_missing.png')

In [49]:

# create a field for predicting employees that miss more than 5 hours
df['miss_more_than_5_hours'] = df['Absenteeism time in hours'].map(lambda hr: 1 if hr > 5 else 0)
df.drop(['Absenteeism time in hours', 'ID'], inplace=True, axis=1)
df

KeyError: "['ID'] not found in axis"

In [30]:
fig = px.pie(df, 'miss_more_than_5_hours', title='37% of workers miss more than 5 hours')
fig.write_image('worker_hours.png')

In [50]:
X = df.drop('miss_more_than_5_hours', axis=1)
y = df['miss_more_than_5_hours']

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, test_size=.2)

In [51]:
pipe = Pipeline([
    ('std', StandardScaler()),
    ('poly', PolynomialFeatures()),
    ('log', LogisticRegression())
])

grid = GridSearchCV(
    pipe,
    param_grid={
        'poly__degree': [2, 3, 4]
    },
    verbose=3
)

grid.fit(X_train, y_train)

Fitting 5 folds for each of 3 candidates, totalling 15 fits
[CV 1/5] END ....................poly__degree=2;, score=0.874 total time=   0.0s
[CV 2/5] END ....................poly__degree=2;, score=0.908 total time=   0.0s
[CV 3/5] END ....................poly__degree=2;, score=0.890 total time=   0.0s
[CV 4/5] END ....................poly__degree=2;, score=0.847 total time=   0.0s
[CV 5/5] END ....................poly__degree=2;, score=0.898 total time=   0.0s
[CV 1/5] END ....................poly__degree=3;, score=0.908 total time=   0.0s
[CV 2/5] END ....................poly__degree=3;, score=0.924 total time=   0.0s
[CV 3/5] END ....................poly__degree=3;, score=0.907 total time=   0.0s
[CV 4/5] END ....................poly__degree=3;, score=0.864 total time=   0.0s
[CV 5/5] END ....................poly__degree=3;, score=0.907 total time=   0.0s
[CV 1/5] END ....................poly__degree=4;, score=0.824 total time=   0.3s
[CV 2/5] END ....................poly__degree=4;,

,estimator,Pipeline(step...egression())])
,param_grid,"{'poly__degree': [2, 3, ...]}"
,scoring,None
,n_jobs,None
,refit,True
,cv,None
,verbose,3
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,copy,True


In [52]:
grid.best_params_

{'poly__degree': 3}

In [53]:
f1 = f1_score(y_test, grid.best_estimator_.predict(X_test))
recall = recall_score(y_test, grid.best_estimator_.predict(X_test))
precision = precision_score(y_test, grid.best_estimator_.predict(X_test))
accuracy = accuracy_score(y_test, grid.best_estimator_.predict(X_test))
print(f"F1 Score: {f1:.3f}")
print(f"Recall: {recall:.3f}")
print(f"Precision: {precision:.3f}")
print(f"Accuracy: {accuracy:.3f}")

F1 Score: 0.879
Recall: 0.887
Precision: 0.870
Accuracy: 0.912


In [54]:
confusion_matrix(y_test, grid.best_estimator_.predict(X_test))

array([[88,  7],
       [ 6, 47]])